### 🛠️ Setup & Initialization

In [ ]:
# %%capturesdsd

import os
import torch
import warnings
from bisect import bisect_left
warnings.filterwarnings("ignore")

from transformers import AutoProcessor
from modeling_bailingmm2 import BailingMM2NativeForConditionalGeneration

def split_model():
    device_map = {}
    world_size = torch.cuda.device_count()
    num_layers = 32
    layer_per_gpu = num_layers // world_size
    layer_per_gpu = [i * layer_per_gpu for i in range(1, world_size + 1)]
    for i in range(num_layers):
        device_map[f'model.model.layers.{i}'] = bisect_left(layer_per_gpu, i)
    device_map['vision'] = 0
    device_map['audio'] = 0
    device_map['linear_proj'] = 0
    device_map['linear_proj_audio'] = 0
    device_map['model.model.word_embeddings.weight'] = 0
    device_map['model.model.norm.weight'] = 0
    device_map['model.lm_head.weight'] = 0
    device_map['model.model.norm'] = 0
    device_map[f'model.model.layers.{num_layers - 1}'] = 0
    device_map['talker'] = 0
    return device_map

# Load pre-trained model with optimized settings, this will take ~10 minutes
model_path = "inclusionAI/Ming-flash-omni-Preview"
model = BailingMM2NativeForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map=split_model(),
    load_image_gen=True,
    load_talker=True,
).to(dtype=torch.bfloat16)

# Initialize processor for handling multimodal inputs
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

[2025-10-14 20:41:59,394] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2025-10-14 20:41:59,456 - root - INFO - gcc -pthread -B /opt/conda/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/conda/include -fPIC -O2 -isystem /opt/conda/include -fPIC -c /tmp/tmp7wdxz6eb/test.c -o /tmp/tmp7wdxz6eb/test.o
2025-10-14 20:41:59,651 - root - INFO - gcc -pthread -B /opt/conda/compiler_compat /tmp/tmp7wdxz6eb/test.o -laio -o /tmp/tmp7wdxz6eb/a.out
2025-10-14 20:41:59,880 - root - INFO - gcc -pthread -B /opt/conda/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /opt/conda/include -fPIC -O2 -isystem /opt/conda/include -fPIC -c /tmp/tmpfetlabj5/test.c -o /tmp/tmpfetlabj5/test.o
2025-10-14 20:41:59,895 - root - INFO - gcc -pthread -B /opt/conda/compiler_compat /tmp/tmpfetlabj5/test.o -L/usr/local/cuda -L/usr/local/cuda/lib64 -lcufile -o /tmp/tmpfetlabj5/a.out
/opt/conda/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try usin

google/byt5-small inclusionAI/Ming-flash-omni-Preview/byt5/./google__byt5-smal/ inclusionAI/Ming-flash-omni-Preview/byt5/./font_uni_10-lang_idx.json inclusionAI/Ming-flash-omni-Preview/byt5/./color_idx.json


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


loaded byt5 base model from inclusionAI/Ming-flash-omni-Preview/byt5
loaded byt5_mapper from inclusionAI/Ming-flash-omni-Preview/byt5
loaded byt5_model from inclusionAI/Ming-flash-omni-Preview/byt5
INFO 10-14 20:43:30 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 10-14 20:43:30 [__init__.py:239] Automatically detected platform cuda.


2025-10-14 20:43:33,288	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


### ⚙️ Inference Pipeline

In [2]:
def generate(messages, processor, model, sys_prompt_exp=None, use_cot_system_prompt=False, max_new_tokens=512):
    text = processor.apply_chat_template(
        messages, 
        sys_prompt_exp=sys_prompt_exp,
        use_cot_system_prompt=use_cot_system_prompt
    )
    image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        audios=audio_inputs,
        return_tensors="pt",
        audio_kwargs={"use_whisper_encoder": True},
    ).to(model.device)

    for k in inputs.keys():
        if k == "pixel_values" or k == "pixel_values_videos" or k == "audio_feats":
            inputs[k] = inputs[k].to(dtype=torch.bfloat16)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            eos_token_id=processor.gen_terminator,
            num_logits_to_keep=1,
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return output_text

In [3]:
import time

def generate_tts(output_text, voice_name='DB30', prompt_text='感谢你的认可。', prompt_wav_path='data/spks/prompt.wav'):
    if model.talker is not None and model.talker_vae is not None:
        model.talker.use_vllm = False
        model.talker.eval()
        model.talker_vae.eval()
        srt_time = time.time()
        all_wavs = []
        with torch.inference_mode():
            for tts_speech, _, _, _ in model.talker.omni_audio_generation(
                    tts_text=output_text,
                    voice_name=voice_name,
                    prompt_text=prompt_text,
                    prompt_wav_path=prompt_wav_path,
                    audio_detokenizer=model.talker_vae, stream=True
            ):
                all_wavs.append(tts_speech)
        waveform = torch.cat(all_wavs, dim=-1)
        print(f"Generate time: {(time.time() - srt_time):.2f}s")
        return waveform.T.numpy(), model.talker_vae.sr

### 💬 Text QA Example

In [4]:
# qa
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "text", "text": "请详细介绍鹦鹉的生活习性。"}
        ],
    },
]

from IPython.utils import io
with io.capture_output() as captured:
    output_text = generate(messages, processor=processor, model=model)

print("请详细介绍鹦鹉的生活习性。\n")
print("Answer:")
print(output_text)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


请详细介绍鹦鹉的生活习性。

Answer:
鹦鹉是一类非常聪明且社交性强的鸟类，它们的生活习性丰富多彩，以下是一些主要的习性：

1. **社交性**：鹦鹉是高度社交的鸟类，它们通常生活在群体中，无论是野外还是人工饲养的环境中。它们通过叫声、羽毛展示和身体语言与同伴交流。

2. **模仿能力**：鹦鹉，尤其是非洲灰鹦鹉，以其惊人的模仿能力而闻名。它们可以模仿人类语言、声音，甚至其他鸟类的叫声。这种能力使它们成为非常受欢迎的宠物。

3. **饮食习惯**：鹦鹉的饮食主要包括种子、果实、坚果、花蜜和昆虫。在野外，它们会根据季节和食物的可获得性调整饮食。在人工饲养的环境中，提供均衡的饮食非常重要，包括专门的鹦鹉饲料、新鲜水果和蔬菜。

4. **筑巢与繁殖**：鹦鹉通常在树洞或岩石缝隙中筑巢。雌鸟会产下2-4个蛋，孵化期大约为2-4周，具体取决于物种。幼鸟在孵化后需要父母的照顾，直到它们能够独立生活。

5. **飞行能力**：鹦鹉是优秀的飞行者，它们的翅膀结构使它们能够进行长距离飞行。在野外，飞行是它们寻找食物、逃避捕食者和迁徙的重要方式。

6. **智力与学习**：鹦鹉具有高度的智力，它们能够解决问题、记忆和学习新技能。这种智力使它们能够适应复杂的环境，也是它们成为受欢迎宠物的原因之一。

7. **情感表达**：鹦鹉能够表达一系列情感，包括快乐、悲伤、愤怒和好奇。它们通过叫声、羽毛展示和身体语言来传达这些情感。

8. **寿命**：鹦鹉的寿命因物种而异，一些小型鹦鹉的寿命可能只有10-15年，而大型鹦鹉如非洲灰鹦鹉和金刚鹦鹉的寿命可以达到50-80年。

了解鹦鹉的生活习性对于它们的饲养和保护至关重要。在人工饲养环境中，提供足够的空间、社交互动、智力刺激和均衡的饮食是确保它们健康和幸福的关键。


### 🖼️ Image QA Example

In [5]:

import os

os.environ["TOKENIZERS_PARALLELISM"] = "true"  # Use to suppress warnings

# image qa
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": "figures/cases/flower.jpg"},
            {"type": "text", "text": "What kind of flower is this?"},
        ],
    },
]

from IPython.utils import io
with io.capture_output() as captured:  # used to suppress output inside the function
    output_text = generate(messages, processor=processor, model=model)

from IPython.display import display, Image
display(Image("figures/cases/flower.jpg"))
print("What kind of flower is this?\n")
print(f"Answer: {output_text}")

What kind of flower is this?

Answer: This is a lotus flower.


<IPython.core.display.Image object>

### 🤔 Chain-of-Thought Reasoning
**To enable thinking before response, adding the following system prompt before your question:**

In [6]:
import os
import json
os.environ["TOKENIZERS_PARALLELISM"] = "true"

question = "Please answer the question and provide the correct option letter, e.g., A, B, C, D, at the end.\nQuestion: Find $m\\angle H$\nChoices:\n(A) 97\n(B) 102\n(C) 107\n(D) 122"
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": "figures/cases/reasoning.png"},
            {"type": "text", "text": question},
        ],
    },
]

from IPython.utils import io
with io.capture_output() as captured: # used to suppress output inside the function
    output_text = generate(messages, processor=processor, model=model, max_new_tokens=8192, use_cot_system_prompt=True)

from IPython.display import display, Image
display(Image("figures/cases/reasoning.png"))
print(f"Input: {question}\n")
print("Answer:\n")
print(output_text)

Input: Please answer the question and provide the correct option letter, e.g., A, B, C, D, at the end.
Question: Find $m\angle H$
Choices:
(A) 97
(B) 102
(C) 107
(D) 122

Answer:

<think>
Got it, let's see. The figure is a pentagon? Wait, no, wait. Wait, the shape here—wait, actually, the sum of interior angles of a polygon. Wait, how many sides? Let's count the vertices: F, G, H, J, E. So that's a pentagon. The formula for the sum of interior angles of an n-sided polygon is (n-2)*180 degrees. So for a pentagon, n=5, so (5-2)*180 = 3*180 = 540 degrees. So the sum of all interior angles is 540°.

Now, let's list the angles:

Angle at F: (x + 20)°

Angle at G: (x + 5)°

Angle at H: (x - 5)°

Angle at J: (x + 10)°

Angle at E: x°

So we need to add all these up and set equal to 540. Let's write that equation:

(x + 20) + (x + 5) + (x - 5) + (x + 10) + x = 540

Let's combine like terms. Let's add the x terms: x + x + x + x + x = 5x

Now the constant terms: 20 + 5 - 5 + 10 = 20 + 5 is 25, m

<IPython.core.display.Image object>

### 🎥 Video QA Example

In [7]:
# video qa
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "video", "video": "figures/cases/yoga.mp4", "max_frames": 64, "sample": "uniform"},
            {"type": "text", "text": "What is the woman doing?"},
        ],
    },
]

from IPython.utils import io
with io.capture_output() as captured: # used to suppress output inside the function
    output_text = generate(messages, processor=processor, model=model)

import io
import base64
from IPython.display import HTML

video = io.open('figures/cases/yoga.mp4', 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''<video alt="test" controls>
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>
             <p>What is the woman doing in the video?</p>
             <p>Answer: {1}</p>
          '''.format(encoded.decode('ascii'), output_text))


2025-10-14 20:47:09,406 - bailingmm_utils - INFO - decord:  video_path='figures/cases/yoga.mp4', total_frames=300, video_fps=30.0, time=0.259s
2025-10-14 20:47:09,474 - bailingmm_utils - INFO - video-in:  num_frames=20, resized_height=308, resized_width=560


### 💬 Multi-turn Conversation

In [8]:
# multi-turn chat
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "text", "text": "中国的首都是哪里？"},
        ],
    },
    {
        "role": "ASSISTANT",
        "content": [
            {"type": "text", "text": "北京"},
        ],
    },
    {
        "role": "HUMAN",
        "content": [
            {"type": "text", "text": "它有哪些著名景点？列出3个。"},
        ],
    },
]

from IPython.utils import io
with io.capture_output() as captured: # used to suppress output inside the function
    output_text = generate(messages, processor=processor, model=model)

print("Human: 中国的首都是哪里？")
print("Assistant: 北京")
print("Human: 它有哪些著名景点？列出3个。\n")

print("Answer:")
print(output_text)

Human: 中国的首都是哪里？
Assistant: 北京
Human: 它有哪些著名景点？列出3个。

Answer:
北京有许多著名的景点，以下是其中三个：

1. 故宫：位于北京市中心，是明清两代的皇宫，也是世界上现存规模最大、保存最完整的木质结构古建筑群。

2. 长城：位于北京市郊，是中国古代伟大的防御工程，也是世界文化遗产之一。

3. 天坛：位于北京市南部，是明清两代皇帝祭天、祈谷的地方，也是世界文化遗产之一。


### 🔊 Audio Tasks

For detailed usage for ASR, SpeechQA, and TTS tasks, please refer to `test_audio_tasks.py`

In [9]:
# ASR
from IPython.display import display, Audio
print("Testing ASR...")
messages = [
    {
        "role": "HUMAN",
        "content": [
            {
                "type": "text",
                "text": "Please recognize the language of this speech and transcribe it. Format: oral.",
            },
            {"type": "audio", "audio": "data/wavs/BAC009S0915W0283.wav"},
        ],
    },
]

output_text = generate(messages=messages, model = model, processor = processor)
output_text = output_text.split('\t', maxsplit=1)[1]

print("Input audio:")
display(Audio('data/wavs/BAC009S0915W0283.wav'))
print(f"Output text: {output_text}")

Testing ASR...
Input audio:


Output text: 也正是看中了中国消费者的消费潜力


In [10]:
# Speech QA

print("Testing Speech QA...")
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "audio", "audio": "data/wavs/speechQA_sample.wav"},
        ],
    },
]

text_response = generate(messages=messages, model = model, processor = processor)

print("Input audio:")
display(Audio('data/wavs/speechQA_sample.wav'))

print(f"Output text: {text_response}")

Testing Speech QA...
Input audio:


Output text: 我是一个多模态模型，名为百灵-原生多模态。我由蚂蚁集团研发，旨在为用户提供有帮助、诚实、无害的回答。我能够处理文本、图片、视频、音频等多种模态的信息输入，并能够进行图文处理、文本生成、文本分析等任务。我基于Transformer架构的最新版本语言模型，拥有广泛的知识、多语言支持、持续学习与更新等优点。


In [11]:

output_audio_path = 'data/wavs/out_tts.wav'
# TTS Generation
import soundfile

print("Testing TTS...")
tts_text="这是一条测试语句。"
audio_wavforms, sr = generate_tts(tts_text, 
        voice_name='DB30',
        prompt_text='感谢你的认可。', 
        prompt_wav_path='data/spks/prompt.wav')
soundfile.write(f"{output_audio_path}", audio_wavforms, sr)
display(Audio(output_audio_path))

Testing TTS...


2025-10-14 20:47:30,100 - root - INFO - register_prompt_wav with 感谢你的认可。, data/spks/prompt.wav


execute llm_job
Generate time: 10.22s


### 🎨 Image Generation & Editing

Ming-omni natively supports image generation and image editing. To use this function, you only need to add the corresponding parameters in the generate function.

In [12]:
# Image generation mode currently limits the range of input pixels.
gen_input_pixels = 451584
processor.image_processor.max_pixels = gen_input_pixels
processor.image_processor.min_pixels = gen_input_pixels

from PIL import Image as PIL_Image

def resize_to_512(image):
    """
    将PIL.Image对象调整到最长边为512像素，保持长宽比不变
    
    参数:
    image (PIL.Image): 输入的PIL图像对象
    
    返回:
    PIL.Image: 调整后的图像对象
    """
    width, height = image.size
    # 计算缩放比例（基于最长边）
    scale = 512 / max(width, height)
    new_width = int(width * scale)
    new_height = int(height * scale)
    # 使用高质量的LANCZOS插值进行缩放
    return image.resize((new_width, new_height), PIL_Image.LANCZOS)

<p data-lake-id="uaf1cb6fa" id="uaf1cb6fa"><span data-lake-id="ud4a6955a" id="ud4a6955a">Generate text in image.</span></p>

In [13]:
from IPython.display import display, Image

prompt =  "A whimsical comic-style illustration of a cozy bookstore entrance on a sunny afternoon. The storefront features warm brick walls and large glass windows filled with stacked books and potted ferns. Above the wooden door hangs a hand-painted signboard with bold, stylized Chinese characters reading “理解与生成统一” accented with curling vines and tiny stars. Sunlight casts playful shadows on the cobblestone path leading to the door, where a vintage lantern in a sunbeam add charm. The linework is clean, colors vibrant yet soft, evoking a friendly, storybook atmosphere. No people or vehicles are present, emphasizing quiet serenity."

messages = [
    {
        "role": "HUMAN",
        "content": [          
            {"type": "text", "text": prompt},
        ],
    }
]
image_save_path = "./bookshore.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=5,
)
image.save(image_save_path)
print("Generated image :")
display(Image(image_save_path))


messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": "./bookshore.jpg"},
            {"type": "text", "text": "将文字内容 替换成 “理解与生成促进”"},
        ],
    }
]
image_save_path = "./bookshore_edit.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=5,
)
image.save(image_save_path)

print("Edit text :")
display(Image(image_save_path))





image_gen_seed:  5
image_gen_steps:  30
image_gen_height:  512
image_gen_width:  512
image_gen_text:  ['Text "理解与生成统一". ']
condition_embeds.shape:  torch.Size([1, 512, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:02<00:00, 11.23it/s]


Generated image :


<IPython.core.display.Image object>

/opt/conda/lib/python3.10/site-packages/transformer_engine/pytorch/module/base.py:592: DeprecationWarning: torch.get_autocast_gpu_dtype() is deprecated. Please use torch.get_autocast_dtype('cuda') instead. (Triggered internally at /pytorch/torch/csrc/autograd/init.cpp:787.)
  self.activation_dtype = torch.get_autocast_gpu_dtype()


image_gen_seed:  5
image_gen_steps:  30
image_gen_height:  512
image_gen_width:  512
image_gen_text:  ['Text "理解与生成促进". ']
condition_embeds.shape:  torch.Size([1, 512, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:04<00:00,  6.84it/s]


Edit text :


<IPython.core.display.Image object>

<p data-lake-id="uc952eed1" id="uc952eed1"><span data-lake-id="u85f40c4d" id="u85f40c4d">Landmark photo generation</span></p>

In [14]:
input_image_path = "./figures/cases/tourist_dessert.png"
print("Reference phote :")
display(resize_to_512(PIL_Image.open(input_image_path)))

messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": input_image_path},
            {"type": "text", "text": "change the background to The Coliseum"},
        ],
    }
]
image_save_path = "./tourist_dessert_change_bg.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=42,
)
image.save(image_save_path)

print("Change background :")
display(Image(image_save_path))

Reference phote :


<PIL.Image.Image image mode=RGB size=384x512>

/opt/conda/lib/python3.10/site-packages/transformer_engine/pytorch/module/base.py:592: DeprecationWarning: torch.get_autocast_gpu_dtype() is deprecated. Please use torch.get_autocast_dtype('cuda') instead. (Triggered internally at /pytorch/torch/csrc/autograd/init.cpp:787.)
  self.activation_dtype = torch.get_autocast_gpu_dtype()


image_gen_seed:  42
image_gen_steps:  30
image_gen_height:  576
image_gen_width:  448
image_gen_text:  ['']
condition_embeds.shape:  torch.Size([1, 256, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:03<00:00,  7.56it/s]


Change background :


<IPython.core.display.Image object>

<p data-lake-id="u6b11b161" id="u6b11b161"><span data-lake-id="u0ecb0564" id="u0ecb0564">Segmentation</span></p>

In [15]:
messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": "./figures/cases/tourist_dessert.png"},
            {"type": "text", "text": "please segment the hair using red mask"},
        ],
    }
]
image_save_path = "./tourist_dessert_segment_hair.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=42,
)
image.save(image_save_path)

print("Segment hair using red mask :")
display(Image(image_save_path))



image_gen_seed:  42
image_gen_steps:  30
image_gen_height:  576
image_gen_width:  448
image_gen_text:  ['']
condition_embeds.shape:  torch.Size([1, 256, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:03<00:00,  7.56it/s]


Segment hair using red mask :


<IPython.core.display.Image object>

<p data-lake-id="ub59c2947" id="ub59c2947"><span data-lake-id="ufd5fa84e" id="ufd5fa84e">Life photo to ID photo</span></p>

In [16]:
input_image_path = "./figures/cases/lifephoto.png"
print("Life phote :")
display(resize_to_512(PIL_Image.open(input_image_path)))

messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": input_image_path},
            {"type": "text", "text": "Generate a high-resolution, standard ID photo. The subject faces the camera directly with a neutral expression, looking forward,shoulders and above only. Dress the person in formal business attire (dark suit jacket over a crisp white shirt). Remove the background completely, leaving a clean, plain white backdrop."},
        ],
    }
]
image_save_path = "./id_photo.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=42,
)
image.save(image_save_path)

print("ID phote :")
display(Image(image_save_path))


Life phote :


<PIL.Image.Image image mode=RGB size=380x512>

image_gen_seed:  42
image_gen_steps:  30
image_gen_height:  576
image_gen_width:  416
image_gen_text:  ['']
condition_embeds.shape:  torch.Size([1, 256, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:03<00:00,  7.93it/s]


ID phote :


<IPython.core.display.Image object>

<p data-lake-id="u8115d750" id="u8115d750"><span data-lake-id="u5cfeb85a" id="u5cfeb85a">Input multiple image  for editing</span></p>

In [17]:
input_image_path_0 = "./figures/cases/edit_multi_input_0.jpg"
input_image_path_1 = "./figures/cases/edit_multi_input_1.jpg"
input_image_path_2 = "./figures/cases/edit_multi_input_2.jpg"
print("Input 1 / 3 :")
display(resize_to_512(PIL_Image.open(input_image_path_0)))
print("Input 2 / 3 :")
display(resize_to_512(PIL_Image.open(input_image_path_1)))
print("Input 3 / 3 :")
display(resize_to_512(PIL_Image.open(input_image_path_2)))


messages = [
    {
        "role": "HUMAN",
        "content": [
            {"type": "image", "image": input_image_path_0},
            {"type": "image", "image": input_image_path_1},
            {"type": "image", "image": input_image_path_2},
            {"type": "text", "text": "Place the person standing among sunflowers; position the animal lying nearby naturally in the scene."},
        ],
    }
]
image_save_path = "./multi_input_edit.jpg"

text = processor.apply_chat_template(messages, add_generation_prompt=True)
image_inputs, video_inputs, audio_inputs = processor.process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    audios=audio_inputs,
    return_tensors="pt",
).to(model.device)

for k in inputs.keys():
    if k in ["pixel_values", "pixel_values_videos", "audio_feats", "pixel_values_reference"]:
        inputs[k] = inputs[k].to(dtype=torch.bfloat16)

# set `image_gen=True` to enable image generation
image = model.generate(
    **inputs,
    image_gen=True,
    image_gen_seed=42,
)
image.save(image_save_path)

print("Edited :")
display(Image(image_save_path))


Input 1 / 3 :
Input 2 / 3 :
Input 3 / 3 :


<PIL.Image.Image image mode=RGB size=512x339>

<PIL.Image.Image image mode=RGB size=400x512>

/opt/conda/lib/python3.10/site-packages/transformer_engine/pytorch/module/base.py:592: DeprecationWarning: torch.get_autocast_gpu_dtype() is deprecated. Please use torch.get_autocast_dtype('cuda') instead. (Triggered internally at /pytorch/torch/csrc/autograd/init.cpp:787.)
  self.activation_dtype = torch.get_autocast_gpu_dtype()


<PIL.Image.Image image mode=RGB size=512x341>

image_gen_seed:  42
image_gen_steps:  30
image_gen_height:  576
image_gen_width:  448
image_gen_text:  ['']
condition_embeds.shape:  torch.Size([1, 256, 2048])
[Hyper-Param]: 
   do_img_cfg: False 
   guidance_scale: 5.0 
   img_cfg: 1.0 
   cfg_mode: 1


100%|██████████| 30/30 [00:03<00:00,  7.56it/s]


Edited :


<IPython.core.display.Image object>